# Module 4 — Q&A RAG Pipeline

**Goal:** given a customer question, retrieve the most relevant knowledge-base entries
and have an LLM generate a grounded, natural-language answer — instead of the LLM
hallucinating policy details (refund windows, shipping options, account rules...) from
its own memory.

**Knowledge base:** we reuse the same Bitext customer-support dataset from Module 3.
Each row already has an `instruction` (a customer question) paired with a curated
`response` (the correct virtual-assistant answer) for one of 27 intents — this is
effectively a ready-made FAQ knowledge base, so no separate document collection is
needed.

**Design decisions (documented for the assessment):**
- **Embeddings — `sentence-transformers/all-MiniLM-L6-v2`**: small (80MB), fast on CPU,
  and a strong general-purpose baseline for semantic similarity — appropriate for a
  chatbot that needs low-latency retrieval on every message.
- **Vector DB — FAISS** (`IndexFlatIP` over L2-normalized vectors = cosine similarity):
  chosen over Qdrant for this deliverable because it needs no separate server process —
  everything runs in-process, which keeps the deployment script simple. The retrieval
  logic is DB-agnostic (embed → search → get top-k ids), so swapping in Qdrant later is
  a drop-in change if persistence/scale requirements grow.
- **Knowledge-base deduplication:** many rows share the same `intent` with near-duplicate
  `response` text (the dataset was generated with linguistic variation on the *question*
  side, not the answer side). We deduplicate to **one representative response per
  intent** so the index doesn't return 20 near-identical chunks for the same topic.
- **Generation — Groq LLM (`openai/gpt-oss-20b`)**: fast inference, and we constrain the
  prompt to answer *only* from the retrieved context, with an explicit fallback
  ("I don't have that information — let me connect you with a human agent") to reduce
  hallucination on out-of-scope questions.


In [1]:
# 1. Imports
import os
import numpy as np
import pandas as pd
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from groq import Groq

RANDOM_STATE = 42


e:\pytorch-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. Load the knowledge base (same dataset as Module 3)
raw = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = raw["train"].to_pandas()
df["category"] = df["category"].astype(str).str.strip().str.upper()
df.shape


(26872, 5)

## 2.1 Build a deduplicated knowledge base

One representative `(instruction, response)` pair per **intent** — this gives the
retriever 27 clean, non-redundant knowledge chunks instead of ~27,000 near-duplicates.

In [3]:
# 3. One representative document per intent
kb_df = (
    df.groupby("intent")
    .first()[["category", "instruction", "response"]]
    .reset_index()
)
kb_df["doc_text"] = (
    "Category: " + kb_df["category"]
    + " | Question: " + kb_df["instruction"]
    + " | Answer: " + kb_df["response"]
)
print(f"Knowledge base size: {len(kb_df)} documents (one per intent)")
kb_df.head()


Knowledge base size: 27 documents (one per intent)


,intent,category,instruction,response,doc_text
0,cancel_order,ORDER,question about cancelling order {{Order Number}},I've understood you have a question regarding ...,Category: ORDER | Question: question about can...
1,change_order,ORDER,I try to change several bloody items of order ...,We understand that you would like to make chan...,Category: ORDER | Question: I try to change se...
2,change_shipping_address,SHIPPING,give me information about a delivery address m...,I'm happy to help! If you need information on ...,Category: SHIPPING | Question: give me informa...
3,check_cancellation_fee,CANCEL,"I can't ifnd the bloody termination charge, I ...",No worries! I'll assist you in finding the ter...,Category: CANCEL | Question: I can't ifnd the ...
4,check_invoice,INVOICE,show me invoice{{Invoice Number}},I understand your need to locate your bill wit...,Category: INVOICE | Question: show me invoice{...


## 3.1 Embed the knowledge base

We embed the `doc_text` (question + answer + category, concatenated) so retrieval can
match on either how the question was phrased or what the answer is about.

In [4]:
# 4. Load the embedding model and encode the knowledge base
embedder = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = embedder.encode(
    kb_df["doc_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # pre-normalize so inner product == cosine similarity
)
kb_embeddings.shape


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]


(27, 384)

In [5]:
# 5. Build the FAISS index (flat inner-product index over normalized vectors)
embedding_dim = kb_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(kb_embeddings)
print(f"FAISS index built: {index.ntotal} vectors, dim={embedding_dim}")


FAISS index built: 27 vectors, dim=384


## 5.1 Retrieval function

In [6]:
# 6. Retrieval: embed the query, search top-k
def retrieve(query: str, top_k: int = 3):
    query_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    scores, indices = index.search(query_emb, top_k)
    results = kb_df.iloc[indices[0]].copy()
    results["score"] = scores[0]
    return results[["intent", "category", "instruction", "response", "score"]]

# sanity check
retrieve("how do I get my money back for a bad product?", top_k=3)


,intent,category,instruction,response,score
6,check_refund_policy,REFUND,i do not know hoow i could check ur reimbursem...,I'm cognizant of the fact that you're unsure a...,0.416598
16,get_refund,REFUND,how do I get a compensation of my money?,I can see that you're seeking guidance on how ...,0.407038
18,payment_issue,PAYMENT,i cant make a payment where can i notify of an...,I'm sorry to hear that you're having trouble m...,0.304495


## 6.1 Generation with Groq

The prompt explicitly tells the model to rely **only** on the retrieved context, and to
say so honestly (routing to a human) rather than guessing when the context doesn't
cover the question. This is the single most important line for keeping the chatbot from
inventing policy details.

Set your API key as an environment variable before running this cell:
`export GROQ_API_KEY="your-key-here"` (never hardcode it in the notebook).

In [24]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

key = os.getenv("GROQ_API_KEY")

print("Loaded:", key is not None)
print("Starts with gsk_:", key.startswith("gsk_") if key else False)
print("Length:", len(key) if key else 0)
print("First 4:", key[:4] if key else None)

Loaded: True
Starts with gsk_: True
Length: 56
First 4: gsk_


In [25]:
from dotenv import load_dotenv
import os

load_dotenv()

print("API key loaded:", os.getenv("GROQ_API_KEY") is not None)

API key loaded: True


In [26]:
# 7. Groq client + prompt template
client = Groq(api_key=os.environ["GROQ_API_KEY"])

SYSTEM_PROMPT = """You are a helpful e-commerce customer support assistant.
Answer the customer's question using ONLY the information in the CONTEXT below.
If the context does not contain enough information to answer confidently, say you
don't have that information and offer to connect the customer with a human agent.
Keep answers concise, polite, and to the point. Do not invent policies, prices,
order numbers, or timelines that are not present in the context."""

def build_prompt(query: str, retrieved_docs: pd.DataFrame) -> str:
    context = "\n\n".join(
        f"[{row.category} / {row.intent}]\nQ: {row.instruction}\nA: {row.response}"
        for row in retrieved_docs.itertuples()
    )
    return f"""CONTEXT:
{context}

CUSTOMER QUESTION:
{query}

Answer the customer's question based on the context above."""


In [27]:
# 8. End-to-end RAG function
def answer_question(query: str, top_k: int = 3, model: str = "openai/gpt-oss-20b") -> str:
    retrieved_docs = retrieve(query, top_k=top_k)
    user_prompt = build_prompt(query, retrieved_docs)

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=300,
    )
    return completion.choices[0].message.content


In [28]:
# 9. Try it out
test_questions = [
    "How can I get a refund for my order?",
    "I want to change the address my package is being delivered to.",
    "Can I speak to a real person about my complaint?",
    "Do you sell flights to Paris?",   # out-of-scope, should trigger the fallback
]

for q in test_questions:
    print("Q:", q)
    print("A:", answer_question(q))
    print("-" * 80)


Q: How can I get a refund for my order?
A: I’m happy to help you with a refund. Could you let me know the details of the situation that led to your request?
--------------------------------------------------------------------------------
Q: I want to change the address my package is being delivered to.
A: Sure! To update the delivery address, just let me know the new address details and I’ll make sure your package is redirected there.
--------------------------------------------------------------------------------
Q: Can I speak to a real person about my complaint?
A: Yes—our customer support team is available to talk with you in person. You can reach a representative during our customer support hours at {{Customer Support Hours}}. Feel free to call or chat then, and they’ll help you with your complaint.
--------------------------------------------------------------------------------
Q: Do you sell flights to Paris?
A: I’m sorry, but I don’t have information about selling flights to Pa

## 10. Persist the retrieval artifacts for deployment

The generation step (Groq call) needs no saved state — only the FAISS index, the
knowledge-base dataframe, and the embedding model name need to ship to the API service.

In [29]:
# 10. Save retrieval artifacts
faiss.write_index(index, "rag_knowledge_base.index")
kb_df.to_csv("rag_knowledge_base.csv", index=False)

print("Saved: rag_knowledge_base.index, rag_knowledge_base.csv")
print("Embedding model to reload at inference time: all-MiniLM-L6-v2")


Saved: rag_knowledge_base.index, rag_knowledge_base.csv
Embedding model to reload at inference time: all-MiniLM-L6-v2
